In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import cross_validate
from xgboost import XGBClassifier

In [46]:
import json

season_tables = []

for season in ["2023_24", "2024_25", "2025_26"]:
    filename = f"premier_league_{season}.json"

    with open(filename, encoding="utf-8") as file:
        data = json.load(file)

    season_matches = pd.json_normalize(data["matches"])
    season_matches["season"] = season
    season_tables.append(season_matches)

matches = pd.concat(season_tables, ignore_index=True)

matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values("date").reset_index(drop=True)
# Use the alternative score column wherever score.ft is missing.
matches["score.ft"] = matches["score.ft"].fillna(matches["score"])

In [47]:
matches["home_goals"] = matches["score.ft"].str[0]
matches["away_goals"] = matches["score.ft"].str[1]

matches[
    ["date", "team1", "team2", "home_goals", "away_goals"]
].head()

,date,team1,team2,home_goals,away_goals
0,2023-08-11,Burnley FC,Manchester City FC,0,3
1,2023-08-12,Arsenal FC,Nottingham Forest FC,2,1
2,2023-08-12,AFC Bournemouth,West Ham United FC,1,1
3,2023-08-12,Brighton & Hove Albion FC,Luton Town FC,4,1
4,2023-08-12,Everton FC,Fulham FC,0,1


In [48]:
# Confirm every match has both goal totals.
assert matches[["home_goals", "away_goals"]].notna().all().all()

# Collect all team names across the three seasons.
teams = pd.concat([matches["team1"], matches["team2"]]).unique()

In [49]:
def expected_result(home_rating, away_rating):
    # returns the expected score for the home team given their rating and the away team's rating.
    return 1 / (1 + (10 ** ((away_rating - home_rating) / 400)))

In [50]:
def update_ratings(home_rating, away_rating, home_goals, away_goals, k=20):
    # Return the updated home and away ratings.
    exp_result = expected_result(home_rating, away_rating)
    if home_goals > away_goals:
        act_result = 1
    elif home_goals == away_goals:
        act_result = 0.5
    else:
        act_result = 0
    change = k * (act_result - exp_result)
    home_rating = home_rating + change
    away_rating = away_rating - change
    return (home_rating, away_rating)

In [51]:
# Give every team an initial rating.
ratings = {team: 1500.0 for team in teams}

home_elo_before = []
away_elo_before = []

for match in matches.itertuples(index=False):
    # Update the ratings using the result of a match
    home_team = match.team1
    away_team = match.team2
    home_elo_before.append(ratings[home_team])
    away_elo_before.append(ratings[away_team])
    ratings[home_team], ratings[away_team] = update_ratings(home_elo_before[-1], 
                                        away_elo_before[-1], 
                                        match.home_goals,
                                        match.away_goals)



In [52]:
# Create new prematch elo columns within the matches DataFrame
matches['home_elo_before'] = home_elo_before
matches['away_elo_before'] = away_elo_before

# Create new prematch predictions based on which team has more elo
matches['elo_difference'] = matches['home_elo_before'] - matches['away_elo_before']

In [53]:
def match_result(home_goals, away_goals):
    # Return a categorical value for the result of a match using scoreline
    if home_goals > away_goals:
        return 0
    elif home_goals == away_goals:
        return 1
    elif home_goals < away_goals:
        return 2

In [54]:
# Create match result column for model predictions 
matches['result'] = (matches[['home_goals', 'away_goals']]
                     .apply(lambda x: match_result(x.home_goals, 
                                                   x.away_goals), 
                                                   axis=1))

In [55]:
# Validates new columns to ensure all values are present
matches[['home_elo_before', 'away_elo_before', 'elo_difference', 'result']].isna().sum()

# Validates that each fixture per season only occured once (accounting for home/away)
matches.duplicated(subset=['season', 'team1', 'team2']).sum()

# Ensures that the date column never decreases to ensure matches are chronological
matches['date'].is_monotonic_increasing

True

# XGBoost Model Training

In [56]:
# Creates a dictionary of hyperparameter values for the model to tune to
param_grid = {
    'n_estimators':[100, 300],
    'learning_rate':[0.03, 0.1],
    'max_depth':[2, 3, 4],
    'min_child_weight':[1, 5]
}

# Creates a time series split with 3 splits
tscv = TimeSeriesSplit(n_splits=3)

# Creates a model for every hyperparameter configuration and returns best cross-validated score
search = GridSearchCV(
    estimator = XGBClassifier(n_jobs=4, random_state=0),
    param_grid=param_grid,
    scoring='neg_log_loss',
    cv=tscv,
    n_jobs=1,
    verbose=1,
    refit=True
)